### Setting up the default values and paths

In [0]:
CATALOG = "project"
SCHEMA = "bronze"
VOLUME_PATH = "/Volumes/project/bronze/original_data"

# Use catalog & schema
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"Using catalog: {CATALOG}, schema: {SCHEMA}")

Using catalog: project, schema: bronze


### Load CSV files and create delta tables:

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
def load_csv_to_delta(table_name, csv_filename):
    """
    Load CSV from volume and create Delta table
    
    Args:
        table_name: Name for Delta table (e.g., 'bronze_customers')
        csv_filename: Filename in /Volumes/dai/project/bronze/ (e.g., 'olist_customers_dataset.csv')
    """
    csv_path = f"{VOLUME_PATH}/{csv_filename}"
    
    try:
        # Read CSV with schema inference
        df = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(csv_path)
        
        # Add ingestion metadata
        df = df.withColumn("ingestion_timestamp", current_timestamp())
        
        # Write to Delta table with schema enforcement
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("mergeSchema", "false") \
            .saveAsTable(table_name)
        
        row_count = df.count()
        print(f"✅ {table_name}: {row_count:,} rows")
        return True
        
    except Exception as e:
        print(f"❌ Error loading {table_name}: {str(e)}")
        return False

### Loading all tables:

In [0]:
print("\n" + "="*70)
print("LOADING CSV FILES FROM VOLUME")
print("="*70)

tables = {
    "bronze_customers": "olist_customers_dataset.csv",
    "bronze_orders": "olist_orders_dataset.csv",
    "bronze_order_items": "olist_order_items_dataset.csv",
    "bronze_order_payments": "olist_order_payments_dataset.csv",
    "bronze_order_reviews": "olist_order_reviews_dataset.csv",
    "bronze_sellers": "olist_sellers_dataset.csv",
    "bronze_products": "olist_products_dataset.csv",
    "bronze_product_category_translation": "product_category_name_translation.csv",
    "bronze_geolocation": "olist_geolocation_dataset.csv",
}

results = {}
for table_name, csv_file in tables.items():
    results[table_name] = load_csv_to_delta(table_name, csv_file)



LOADING CSV FILES FROM VOLUME
✅ bronze_customers: 99,441 rows
✅ bronze_orders: 99,441 rows
✅ bronze_order_items: 112,650 rows
✅ bronze_order_payments: 103,886 rows
✅ bronze_order_reviews: 104,162 rows
✅ bronze_sellers: 3,095 rows
✅ bronze_products: 32,951 rows
✅ bronze_product_category_translation: 71 rows
✅ bronze_geolocation: 1,000,163 rows


In [0]:
# tables_df = spark.sql("""
#     SHOW TABLES IN project.bronze
# """)

# olist_tables = tables_df.filter(tables_df.tableName.startswith("olist"))

# for row in olist_tables.collect():
#     table_full_name = f"project.bronze.{row.tableName}"
#     spark.sql(f"DROP TABLE IF EXISTS {table_full_name}")